# Physiologically-Constrained Neural Operator for Heart Sound Classification

**Classes:** AS (Aortic Stenosis) · MR (Mitral Regurgitation) · MS (Mitral Stenosis) · MVP (Mitral Valve Prolapse) · Normal  
**Architecture:** Fourier Neural Operator (FNO) backbone + 5-class classifier head  
**Constraints enforced:**
1. Cardiac cycle periodicity regularisation
2. Frequency band hierarchy (S1 low-freq, S2 high-freq)
3. Physiological frequency mask initialised from S1/S2 acoustics

**Baseline:** Mel-spectrogram CNN  
**Framework:** PyTorch

---
> *"We encode known structural properties of cardiac acoustics as architectural inductive biases,*  
> *analogous to physics-consistent neural architectures in continuum mechanics."*

## 0. Imports and Configuration

In [12]:
# import subprocess
# import sys

# # Install librosa and required dependencies
# print("Installing required audio processing library...")
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "librosa", "soundfile"])
# print("✓ librosa and soundfile installed successfully")

In [13]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q librosa soundfile


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os
import numpy as np
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import librosa
import warnings
warnings.filterwarnings('ignore')

os.makedirs('figures',     exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Path to dataset root ────────────────────────────────────────────────────
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/Heart-Sound/Data'

CLASS_FOLDERS = ['AS_New', 'MR_New', 'MS_New', 'MVP_New', 'N_New']
CLASS_NAMES   = ['AS',     'MR',     'MS',     'MVP',     'Normal']

# Check if data directory is accessible; if not, use sample data
USE_SAMPLE_DATA = not os.path.isdir(DATA_DIR)


# Set your data path (adjust the folder name to match yours on Drive)
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/Heart-Sound/Data'

import os
print(f"Data dir exists: {os.path.exists(DATA_DIR)}")
print(f"Contents: {os.listdir(DATA_DIR)}")

# Define CONFIG dictionary
CONFIG = {
    'sample_rate': 22050,  # Inferred from 66150 samples / 3.00s = 22050 Hz
    'signal_length': 66150, # Inferred from 'Fixed window used' output
    'use_sample_data': USE_SAMPLE_DATA, # Using the already defined variable
    # 'test_split': 0.15, # Common split percentage
    # 'val_split': 0.15,  # Common split percentage
    'test_split': 0.20, # Common split percentage
    'val_split': 0.20,  # Common split percentage
    'batch_size': 32,   # Common batch size
    'n_classes': len(CLASS_NAMES), # Derived from CLASS_NAMES
    'fno_width': 64,    # Common FNO parameter
    # 'fno_modes': 20,    # Common FNO parameter
    'fno_modes': 150,    # Common FNO parameter
    'fno_depth': 4,     # Common FNO parameter
    # 's1_band': [25, 45], # From plot label in 'Learned Frequency Mask' section
    # 's2_band': [50, 70], # From plot label in 'Learned Frequency Mask' section
    'lambda_period': 0.1, # Hyperparameter for periodicity loss
    'lambda_freq': 0.1,   # Hyperparameter for frequency hierarchy loss
    'epochs': 60,       # From train_model print statement 'Ep 1/60'
    'lr': 1e-3,         # Common learning rate
    'weight_decay': 1e-4, # Common weight decay
    'class_names': CLASS_NAMES, # Using the already defined variable
}

# Define sample_signals, as it's used in HeartSoundDataset init
# It is only used if use_synthetic is True. Since USE_SAMPLE_DATA is False,
# setting it to None is appropriate.
sample_signals = None

Device: cpu
Data dir exists: True
Contents: ['MR_New', 'MS_New', 'AS_New', 'N_New', 'MVP_New']


## 1. Dataset

In [15]:
class HeartSoundDataset(Dataset):
    """
    Loads .wav files from CLASS_FOLDERS subfolders OR uses synthetic data if not available.
    Each recording is resampled to CONFIG['sample_rate'],
    trimmed/zero-padded to CONFIG['signal_length'],
    and amplitude-normalised to [-1, 1].
    """
    def __init__(self, data_dir, class_folders,
                 sr=CONFIG['sample_rate'],
                 length=CONFIG['signal_length'],
                 use_synthetic=CONFIG.get('use_sample_data', False),
                 sample_signals=sample_signals):
        self.sr, self.length = sr, length

        if use_synthetic and sample_signals is not None:
            # Use synthetic data
            self.signals = np.array(sample_signals, dtype=np.float32)
            self.labels = np.array([i // (len(sample_signals) // 5) for i in range(len(sample_signals))], dtype=np.int64)
            self.filenames = [f"synthetic_{i}.wav" for i in range(len(sample_signals))]
            print(f'Loaded {len(self.signals)} synthetic recordings')
        else:
            # Load from real files
            self.signals, self.labels, self.filenames = \
                self._load(data_dir, class_folders, sr, length)
            print(f'Loaded {len(self.signals)} real recordings')

        counts = np.bincount(self.labels, minlength=len(class_folders))
        for i, (folder, name) in enumerate(zip(class_folders, CLASS_NAMES)):
            print(f'  [{i}] {name:8s}  ({folder}): {counts[i]:3d} files')

    @staticmethod
    def _load(data_dir, class_folders, sr, length):
        signals, labels, fnames = [], [], []
        for cls_idx, folder in enumerate(class_folders):
            folder_path = os.path.join(data_dir, folder)
            if not os.path.isdir(folder_path):
                print(f'  WARNING: folder not found: {folder_path}')
                continue
            wavs = sorted(f for f in os.listdir(folder_path)
                          if f.lower().endswith('.wav'))
            for fname in wavs:
                fpath = os.path.join(folder_path, fname)
                try:
                    wav, _ = librosa.load(fpath, sr=sr, mono=True)
                except Exception as e:
                    print(f'  WARNING: skipping {fpath}: {e}')
                    continue
                # Trim or zero-pad
                if len(wav) >= length:
                    wav = wav[:length]
                else:
                    wav = np.pad(wav, (0, length - len(wav)))
                # Amplitude normalise
                mx = np.abs(wav).max()
                if mx > 0:
                    wav /= mx
                signals.append(wav.astype(np.float32))
                labels.append(cls_idx)
                fnames.append(fpath)
        if not signals:
            raise RuntimeError(
                'No .wav files loaded. Check DATA_DIR and CLASS_FOLDERS.')
        return (np.array(signals, dtype=np.float32),
                np.array(labels,  dtype=np.int64),
                fnames)

    def __len__(self):  return len(self.signals)

    def __getitem__(self, idx):
        return (torch.tensor(self.signals[idx]).unsqueeze(0),  # (1, L)
                torch.tensor(self.labels[idx]))


dataset = HeartSoundDataset(DATA_DIR, CLASS_FOLDERS)

n_total = len(dataset)
n_test  = int(n_total * CONFIG['test_split'])
n_val   = int(n_total * CONFIG['val_split'])
n_train = n_total - n_val - n_test

train_ds, val_ds, test_ds = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                          shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'],
                          shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'],
                          shuffle=False)
print(f'\nSplit  Train: {n_train}  |  Val: {n_val}  |  Test: {n_test}')

Loaded 1000 real recordings
  [0] AS        (AS_New): 200 files
  [1] MR        (MR_New): 200 files
  [2] MS        (MS_New): 200 files
  [3] MVP       (MVP_New): 200 files
  [4] Normal    (N_New): 200 files

Split  Train: 600  |  Val: 200  |  Test: 200


In [ ]:
dataset = HeartSoundDataset(DATA_DIR, CLASS_FOLDERS)
print(f"Signal shape: {dataset.signals.shape}")   # should be (N, 66150)
print(f"Label distribution: {np.bincount(dataset.labels)}")

## 2. Exploratory Data Analysis

In [ ]:
# ── 2c. Recording duration statistics ───────────────────────────────────────
print('Computing original recording durations...')
if CONFIG.get('use_sample_data', False):
    print('  (Using synthetic data — durations are fixed at 3.0s)')
    print(f'  Min  : {3.0:.2f} s')
    print(f'  Max  : {3.0:.2f} s')
    print(f'  Mean : {3.0:.2f} s +/- {0.0:.2f} s')
else:
    durations = np.array([
        librosa.get_duration(path=f) for f in dataset.filenames])
    print(f'  Min  : {durations.min():.2f} s')
    print(f'  Max  : {durations.max():.2f} s')
    print(f'  Mean : {durations.mean():.2f} s +/- {durations.std():.2f} s')
print(f'  Fixed window used: {dataset.length/dataset.sr:.2f} s ({dataset.length} samples)')

## 3. FNO Architecture

### 3.1 FNO Backbone

| Component | Physics motivation |
|-----------|-------------------|
| Spectral convolution | Frequency-domain operation — natural for periodic PCG signals |
| Mode truncation | Band-limited inductive bias; cardiac acoustics < 1 kHz |
| Bypass Conv1d (1×1) | Local channel mixing; residual path prevents spectral-only bottleneck |
| Global average pooling | Length-invariant representation after pad/trim |

### 3.2 PhysioFrequencyMask

| Parameter | Type | Physics motivation |
|-----------|------|--------------------|
| Per-mode logit `logit[k]` | Learnable (×M) | Free gain per Fourier mode; initialized high in S1/S2 bands |
| S1 band boundaries `s1_lo`, `s1_hi` | Learnable (×2) | Data-driven adaptation of mitral/tricuspid closure band |
| S2 band boundaries `s2_lo`, `s2_hi` | Learnable (×2) | Data-driven adaptation of aortic/pulmonary closure band |
| Smooth bump function | Fixed (double sigmoid) | Differentiable soft band indicator; gradients flow through boundaries |

### 3.3 PhysioFeatureExtractor (18 explicit physical features)

| Feature | Quantity | Learnable component |
|---------|----------|---------------------|
| S1/S2 spectral energy | Energy in each valve closure band | S1/S2 band boundaries |
| S1/S2 spectral centroid | Frequency center of mass per band | S1/S2 band boundaries |
| Murmur bandwidth | Spectral spread of turbulent flow region | Murmur band boundaries |
| Harmonic energies f₀–4f₀ | Energy at fundamental + 3 harmonics | Harmonic weights + HR band |
| Dominant frequency | Heart rate estimate | HR search band boundaries |
| Periodicity ratio | Fraction of energy in cardiac band | Fixed computation |
| RMS amplitude | Overall signal energy | Fixed computation |
| S1/S2 energy ratio | Relative valve closure intensities (log scale) | S1/S2 band boundaries |
| Envelope mean + std | Instantaneous amplitude via Hilbert transform | Fixed computation |
| Instantaneous frequency | Phase derivative of analytic signal | Fixed computation |
| S1-to-S2 interval | Systolic duration proxy (1/2f₀) | HR band boundaries |
| Cross-beat consistency | Harmonic energy ratio across cardiac cycles | HR band boundaries |

### 3.4 Classification Head

| Component | Detail |
|-----------|--------|
| FNO embedding | 64-dim via project MLP |
| Physics features | 18-dim from PhysioFeatureExtractor |
| Fusion | Concatenate → 82-dim |
| Classifier | Linear(82, 5) |

The physics features are not auxiliary supervision — they are direct inputs to the
classifier. The network is explicitly trained to use physical quantities of the
cardiac signal to distinguish valve conditions.

### All class definitions

In [ ]:
def _hz_to_logit(hz, nyquist):
    """Convert a frequency in Hz to the logit of its Nyquist-normalised value."""
    p = np.clip(hz / nyquist, 1e-6, 1 - 1e-6)
    return float(np.log(p / (1 - p)))


class SpectralConv1d(nn.Module):
    """FNO spectral convolution: learn complex weights on retained Fourier modes."""
    def __init__(self, in_ch, out_ch, n_modes):
        super().__init__()
        self.n_modes = n_modes
        scale = 1.0 / (in_ch * out_ch)
        self.wr = nn.Parameter(scale * torch.randn(in_ch, out_ch, n_modes))
        self.wi = nn.Parameter(scale * torch.randn(in_ch, out_ch, n_modes))

    def forward(self, x):
        B, C, L = x.shape
        x_ft   = torch.fft.rfft(x, norm='ortho')
        out_ft = torch.zeros(B, self.wr.shape[1], L//2+1,
                             dtype=torch.cfloat, device=x.device)
        W = torch.complex(self.wr, self.wi)
        out_ft[:, :, :self.n_modes] = torch.einsum(
            'bim,iom->bom', x_ft[:, :, :self.n_modes], W)
        return torch.fft.irfft(out_ft, n=L, norm='ortho')


class PhysioFrequencyMask(nn.Module):
    """
    Learnable soft band-pass mask with two complementary components:

    Component 1 — per-mode logit:
        A free scalar per retained Fourier mode, sigmoid-mapped to (0,1).
        Initialised high (≈1.0) inside S1/S2 bands, low (≈0.1) outside.

    Component 2 — learnable band boundaries:
        Four scalar parameters s1_lo, s1_hi, s2_lo, s2_hi stored as raw
        reals and mapped to Hz via sigmoid(·) * Nyquist. A smooth
        double-sigmoid bump contributes additively to the mask.

    Final mask (per mode k):
        fm[k] = clip(sigmoid(logit[k]) + bump_weight * (B_S1(f_k) + B_S2(f_k)), 0, 1)
    """
    def __init__(self, n_modes, signal_length, sr, s1_band, s2_band,
                 bump_sharpness=50.0, bump_weight=0.3):
        super().__init__()
        self.n_modes        = n_modes
        self.sr             = sr
        self.L              = signal_length
        self.bump_sharpness = bump_sharpness
        self.bump_weight    = bump_weight
        nyquist             = sr / 2.0

        # Component 1: per-mode free logit
        freqs      = np.fft.rfftfreq(signal_length, 1.0 / sr)[:n_modes]
        mask_init  = np.full(n_modes, 0.1)
        for i, f in enumerate(freqs):
            if s1_band[0] <= f <= s1_band[1] or s2_band[0] <= f <= s2_band[1]:
                mask_init[i] = 1.0
        m          = np.clip(mask_init, 1e-6, 1 - 1e-6)
        self.logit = nn.Parameter(
            torch.tensor(np.log(m / (1 - m)), dtype=torch.float32))

        # Component 2: learnable band boundaries
        self.s1_lo = nn.Parameter(torch.tensor(_hz_to_logit(s1_band[0], nyquist)))
        self.s1_hi = nn.Parameter(torch.tensor(_hz_to_logit(s1_band[1], nyquist)))
        self.s2_lo = nn.Parameter(torch.tensor(_hz_to_logit(s2_band[0], nyquist)))
        self.s2_hi = nn.Parameter(torch.tensor(_hz_to_logit(s2_band[1], nyquist)))

    def forward(self, x):
        nyquist  = self.sr / 2.0
        freqs_hz = torch.fft.rfftfreq(self.L, 1.0 / self.sr).to(x.device)
        freqs_n  = freqs_hz / nyquist

        fm = torch.full((freqs_hz.shape[0],), 0.1, device=x.device)
        fm[:self.n_modes] = torch.sigmoid(self.logit)

        k       = self.bump_sharpness
        s1_lo_n = torch.sigmoid(self.s1_lo)
        s1_hi_n = torch.sigmoid(self.s1_hi)
        s2_lo_n = torch.sigmoid(self.s2_lo)
        s2_hi_n = torch.sigmoid(self.s2_hi)
        s1_bump   = (torch.sigmoid(k * (freqs_n - s1_lo_n)) *
                     torch.sigmoid(k * (s1_hi_n  - freqs_n)))
        s2_bump   = (torch.sigmoid(k * (freqs_n - s2_lo_n)) *
                     torch.sigmoid(k * (s2_hi_n  - freqs_n)))
        band_bump = torch.clamp(s1_bump + s2_bump, 0.0, 1.0)

        fm   = torch.clamp(fm + self.bump_weight * band_bump, 0.0, 1.0)
        x_ft = torch.fft.rfft(x, norm='ortho')
        return torch.fft.irfft(
            x_ft * fm.unsqueeze(0).unsqueeze(0),
            n=x.shape[-1], norm='ortho')

    def get_mask(self):
        return torch.sigmoid(self.logit).detach().cpu().numpy()

    def get_bands(self):
        nyquist = self.sr / 2.0
        return {
            'S1': [torch.sigmoid(self.s1_lo).item() * nyquist,
                   torch.sigmoid(self.s1_hi).item() * nyquist],
            'S2': [torch.sigmoid(self.s2_lo).item() * nyquist,
                   torch.sigmoid(self.s2_hi).item() * nyquist],
        }


class FNOBlock1d(nn.Module):
    def __init__(self, width, n_modes):
        super().__init__()
        self.spec   = SpectralConv1d(width, width, n_modes)
        self.bypass = nn.Conv1d(width, width, 1)
        self.norm   = nn.InstanceNorm1d(width)

    def forward(self, x):
        return F.gelu(self.norm(self.spec(x) + self.bypass(x)))


class PhysioFeatureExtractor(nn.Module):
    """
    Extracts explicit physical quantities from a raw PCG signal.

    All quantities are differentiable w.r.t. the input signal.
    Some have learnable parameters (band boundaries, weights).
    Others are fixed physical computations.

    Quantities computed:
      1.  S1/S2 spectral energy          (2 values)
      2.  Spectral centroid S1, S2 bands (2 values)
      3.  Murmur bandwidth               (1 value)
      4.  Harmonic energies f0,2f0,3f0,4f0 (4 values)
      5.  Dominant frequency (heart rate)(1 value)
      6.  Periodicity ratio              (1 value)
      7.  RMS amplitude                  (1 value)
      8.  S1/S2 energy ratio             (1 value)
      9.  Envelope mean and std          (2 values)
      10. Instantaneous frequency mean   (1 value)
      11. S1-to-S2 interval estimate     (1 value)
      12. Cross-beat consistency         (1 value)
    ─────────────────────────────────────────────
    Total:                               18 values
    """
    def __init__(self, signal_length=CONFIG['signal_length'],
                 sr=CONFIG['sample_rate'],
                 n_modes=CONFIG['fno_modes']):
        super().__init__()
        self.sr  = sr
        self.L   = signal_length
        nyquist  = sr / 2.0

        self.s1_lo      = nn.Parameter(torch.tensor(_hz_to_logit(25,  nyquist)))
        self.s1_hi      = nn.Parameter(torch.tensor(_hz_to_logit(45,  nyquist)))
        self.s2_lo      = nn.Parameter(torch.tensor(_hz_to_logit(50,  nyquist)))
        self.s2_hi      = nn.Parameter(torch.tensor(_hz_to_logit(70,  nyquist)))
        self.mu_lo      = nn.Parameter(torch.tensor(_hz_to_logit(100, nyquist)))
        self.mu_hi      = nn.Parameter(torch.tensor(_hz_to_logit(500, nyquist)))
        self.hr_lo      = nn.Parameter(torch.tensor(_hz_to_logit(0.8, nyquist)))
        self.hr_hi      = nn.Parameter(torch.tensor(_hz_to_logit(3.0, nyquist)))
        self.harmonic_w = nn.Parameter(torch.ones(4))

    def _psd_and_freqs(self, s):
        psd = torch.fft.rfft(s, norm='ortho').abs() ** 2
        f   = torch.fft.rfftfreq(self.L, 1.0 / self.sr).to(s.device)
        return psd, f

    def _band_energy(self, psd, f, lo, hi):
        k    = 50.0
        mask = (torch.sigmoid(k * (f - lo)) *
                torch.sigmoid(k * (hi - f)))
        return (psd * mask.unsqueeze(0)).sum(-1)

    def _spectral_centroid(self, psd, f, lo, hi):
        k    = 50.0
        mask = (torch.sigmoid(k * (f - lo)) *
                torch.sigmoid(k * (hi - f)))
        num  = (psd * mask.unsqueeze(0) * f.unsqueeze(0)).sum(-1)
        den  = (psd * mask.unsqueeze(0)).sum(-1) + 1e-8
        return num / den

    def _hilbert_envelope(self, s):
        S    = torch.fft.rfft(s, norm='ortho')
        h    = torch.zeros(S.shape[-1], device=s.device)
        h[0] = 1.0
        if self.L % 2 == 0:
            h[1:self.L//2] = 2.0
            h[self.L//2]   = 1.0
        else:
            h[1:(self.L+1)//2] = 2.0
        analytic = torch.fft.irfft(S * h.unsqueeze(0), n=self.L, norm='ortho')
        return (s ** 2 + analytic ** 2).sqrt()

    def _instantaneous_frequency(self, s):
        S    = torch.fft.rfft(s, norm='ortho')
        h    = torch.zeros(S.shape[-1], device=s.device)
        h[0] = 1.0
        if self.L % 2 == 0:
            h[1:self.L//2] = 2.0
            h[self.L//2]   = 1.0
        else:
            h[1:(self.L+1)//2] = 2.0
        analytic  = torch.fft.irfft(S * h.unsqueeze(0), n=self.L, norm='ortho')
        phase     = torch.atan2(analytic, s)
        return torch.diff(phase, dim=-1) / (2 * math.pi) * self.sr

    def _cross_beat_consistency(self, s, psd, f):
        k     = 50.0
        hr_lo = torch.sigmoid(self.hr_lo) * (self.sr / 2.0)
        hr_hi = torch.sigmoid(self.hr_hi) * (self.sr / 2.0)
        hr_mask = (torch.sigmoid(k * (f - hr_lo)) *
                   torch.sigmoid(k * (hr_hi - f)))
        f0_idx  = (psd * hr_mask.unsqueeze(0)).argmax(-1)
        f0      = f[f0_idx]
        width   = 2.0
        harmonic_energy = torch.zeros(s.shape[0], device=s.device)
        for n in range(1, 5):
            nf0       = (n * f0).unsqueeze(1)
            lo_n      = nf0 - width
            hi_n      = nf0 + width
            band_mask = (torch.sigmoid(k * (f.unsqueeze(0) - lo_n)) *
                         torch.sigmoid(k * (hi_n - f.unsqueeze(0))))
            harmonic_energy += (psd * band_mask).sum(-1)
        return harmonic_energy / (psd.sum(-1) + 1e-8)

    def get_bands(self):
        nyquist = self.sr / 2.0
        return {
            'S1'    : [torch.sigmoid(self.s1_lo).item() * nyquist,
                       torch.sigmoid(self.s1_hi).item() * nyquist],
            'S2'    : [torch.sigmoid(self.s2_lo).item() * nyquist,
                       torch.sigmoid(self.s2_hi).item() * nyquist],
            'murmur': [torch.sigmoid(self.mu_lo).item() * nyquist,
                       torch.sigmoid(self.mu_hi).item() * nyquist],
            'HR'    : [torch.sigmoid(self.hr_lo).item() * nyquist,
                       torch.sigmoid(self.hr_hi).item() * nyquist],
        }

    def forward(self, x):
        s       = x.squeeze(1)
        psd, f  = self._psd_and_freqs(s)
        nyquist = self.sr / 2.0
        k       = 50.0

        s1_lo = torch.sigmoid(self.s1_lo) * nyquist
        s1_hi = torch.sigmoid(self.s1_hi) * nyquist
        s2_lo = torch.sigmoid(self.s2_lo) * nyquist
        s2_hi = torch.sigmoid(self.s2_hi) * nyquist
        mu_lo = torch.sigmoid(self.mu_lo) * nyquist
        mu_hi = torch.sigmoid(self.mu_hi) * nyquist
        hr_lo = torch.sigmoid(self.hr_lo) * nyquist
        hr_hi = torch.sigmoid(self.hr_hi) * nyquist

        # 1. S1/S2 spectral energy
        e_s1 = self._band_energy(psd, f, s1_lo, s1_hi)
        e_s2 = self._band_energy(psd, f, s2_lo, s2_hi)

        # 2. Spectral centroid
        c_s1 = self._spectral_centroid(psd, f, s1_lo, s1_hi)
        c_s2 = self._spectral_centroid(psd, f, s2_lo, s2_hi)

        # 3. Murmur bandwidth
        mu_e    = self._band_energy(psd, f, mu_lo, mu_hi)
        mu_c    = self._spectral_centroid(psd, f, mu_lo, mu_hi)
        mu_mask = (torch.sigmoid(k * (f - mu_lo)) *
                   torch.sigmoid(k * (mu_hi - f)))
        mu_var  = ((psd * mu_mask.unsqueeze(0) *
                    (f.unsqueeze(0) - mu_c.unsqueeze(1)) ** 2).sum(-1) /
                   (mu_e + 1e-8))
        mu_bw   = mu_var.sqrt()

        # 4. Harmonic energies — fully vectorised
        hr_mask = (torch.sigmoid(k * (f - hr_lo)) *
                   torch.sigmoid(k * (hr_hi - f)))
        f0_idx  = (psd * hr_mask.unsqueeze(0)).argmax(-1)
        f0_hz   = f[f0_idx]
        width   = 2.0
        harm_energies = []
        for n in range(1, 5):
            nf0       = (n * f0_hz).unsqueeze(1)
            lo_n      = nf0 - width
            hi_n      = nf0 + width
            band_mask = (torch.sigmoid(k * (f.unsqueeze(0) - lo_n)) *
                         torch.sigmoid(k * (hi_n - f.unsqueeze(0))))
            harm_energies.append((psd * band_mask).sum(-1))
        harm_energies = torch.stack(harm_energies, dim=1)      # (B, 4)

        # 5. Dominant frequency
        f0_feat = f0_hz / nyquist

        # 6. Periodicity ratio
        band   = (f >= 1.0) & (f <= 80.0 / 60 * 4)
        period = psd[:, band].sum(-1) / (psd.sum(-1) + 1e-8)

        # 7. RMS amplitude
        rms = (s ** 2).mean(-1).sqrt()

        # 8. S1/S2 energy ratio
        ratio = e_s1 / (e_s2 + 1e-8)

        # 9. Envelope mean and std
        env      = self._hilbert_envelope(s)
        env_mean = env.mean(-1)
        env_std  = env.std(-1)

        # 10. Instantaneous frequency mean
        if_sig  = self._instantaneous_frequency(s)
        if_mean = if_sig.mean(-1) / nyquist

        # 11. S1-to-S2 interval estimate
        s1s2_interval = 1.0 / (2.0 * f0_hz.clamp(min=0.5))

        # 12. Cross-beat consistency
        consistency = self._cross_beat_consistency(s, psd, f)

        features = torch.stack([
            e_s1, e_s2,
            c_s1 / nyquist, c_s2 / nyquist,
            mu_bw / nyquist,
            harm_energies[:, 0], harm_energies[:, 1],
            harm_energies[:, 2], harm_energies[:, 3],
            f0_feat,
            period,
            rms,
            ratio.log1p(),
            env_mean, env_std,
            if_mean,
            s1s2_interval,
            consistency,
        ], dim=1)                                               # (B, 18)

        return features


class FNOClassifier(nn.Module):
    def __init__(self,
                 n_classes=CONFIG['n_classes'],
                 width=CONFIG['fno_width'],
                 n_modes=CONFIG['fno_modes'],
                 depth=CONFIG['fno_depth'],
                 signal_length=CONFIG['signal_length'],
                 sr=CONFIG['sample_rate']):
        super().__init__()
        self.freq_mask  = PhysioFrequencyMask(
            n_modes, signal_length, sr,
            s1_band=[25, 45], s2_band=[50, 70])
        self.lift       = nn.Conv1d(1, width, 1)
        self.fno_blocks = nn.ModuleList(
            [FNOBlock1d(width, n_modes) for _ in range(depth)])
        self.project    = nn.Sequential(
            nn.Linear(width, 128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, 64),   nn.GELU())
        self.phys       = PhysioFeatureExtractor(signal_length, sr, n_modes)
        self.classifier = nn.Linear(64 + 18, n_classes)

    def forward(self, x):
        h   = self.freq_mask(x)
        h   = self.lift(h)
        for blk in self.fno_blocks:
            h = blk(h)
        h   = h.mean(dim=-1)
        emb = self.project(h)                                  # (B, 64)
        phys = self.phys(x)                                    # (B, 18)
        logits = self.classifier(torch.cat([emb, phys], dim=1))
        return logits, emb

### Instantiation and diagnostics:

In [ ]:
model_fno = FNOClassifier().to(DEVICE)
print(f'FNO  | params: {sum(p.numel() for p in model_fno.parameters()):,}')

nyq = CONFIG['sample_rate'] / 2
print('\nPhysioFrequencyMask (learned mask):')
print(f'  s1_lo: {torch.sigmoid(model_fno.freq_mask.s1_lo).item()*nyq:.1f} Hz')
print(f'  s1_hi: {torch.sigmoid(model_fno.freq_mask.s1_hi).item()*nyq:.1f} Hz')
print(f'  s2_lo: {torch.sigmoid(model_fno.freq_mask.s2_lo).item()*nyq:.1f} Hz')
print(f'  s2_hi: {torch.sigmoid(model_fno.freq_mask.s2_hi).item()*nyq:.1f} Hz')

print('\nPhysioFeatureExtractor (learned physical features):')
print(f'  s1_lo: {torch.sigmoid(model_fno.phys.s1_lo).item()*nyq:.1f} Hz')
print(f'  s1_hi: {torch.sigmoid(model_fno.phys.s1_hi).item()*nyq:.1f} Hz')
print(f'  s2_lo: {torch.sigmoid(model_fno.phys.s2_lo).item()*nyq:.1f} Hz')
print(f'  s2_hi: {torch.sigmoid(model_fno.phys.s2_hi).item()*nyq:.1f} Hz')
print(f'  mu_lo: {torch.sigmoid(model_fno.phys.mu_lo).item()*nyq:.1f} Hz')
print(f'  mu_hi: {torch.sigmoid(model_fno.phys.mu_hi).item()*nyq:.1f} Hz')
print(f'  hr_lo: {torch.sigmoid(model_fno.phys.hr_lo).item()*nyq:.2f} Hz')
print(f'  hr_hi: {torch.sigmoid(model_fno.phys.hr_hi).item()*nyq:.2f} Hz')

## 4. CNN Baseline (Mel Spectrogram)

In [ ]:
class MelSpectrogramLayer(nn.Module):
    """Differentiable log-Mel spectrogram — no librosa at train time."""
    def __init__(self, sr=CONFIG['sample_rate'],
                 n_fft=512, hop=128, n_mels=64):
        super().__init__()
        self.n_fft, self.hop = n_fft, hop
        self.register_buffer('window', torch.hann_window(n_fft))
        self.register_buffer('mel_fb',
            torch.tensor(self._mel_fb(sr, n_fft, n_mels), dtype=torch.float32))

    @staticmethod
    def _mel_fb(sr, n_fft, n_mels, fmin=20, fmax=None):
        fmax = fmax or sr / 2.0
        h2m  = lambda f: 2595 * np.log10(1 + f / 700)
        m2h  = lambda m: 700 * (10 ** (m / 2595) - 1)
        mels  = np.linspace(h2m(fmin), h2m(fmax), n_mels + 2)
        hz    = m2h(mels)
        freqs = np.fft.rfftfreq(n_fft, 1.0/sr)
        fb    = np.zeros((n_mels, len(freqs)))
        for m in range(n_mels):
            lo, c, hi = hz[m], hz[m+1], hz[m+2]
            up   = (freqs >= lo) & (freqs <= c)
            down = (freqs > c)  & (freqs <= hi)
            fb[m, up]   = (freqs[up]   - lo) / (c  - lo + 1e-8)
            fb[m, down] = (hi - freqs[down]) / (hi - c  + 1e-8)
        return fb

    def forward(self, x):
        x    = x.squeeze(1)
        stft = torch.stft(x, n_fft=self.n_fft, hop_length=self.hop,
                          window=self.window, return_complex=True)
        mel  = torch.log(torch.matmul(self.mel_fb, stft.abs()**2) + 1e-6)
        return mel.unsqueeze(1)


class CNNBaseline(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.mel = MelSpectrogramLayer()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, n_classes))

    def forward(self, x):
        return self.head(self.cnn(self.mel(x)))


model_cnn = CNNBaseline().to(DEVICE)
print(f'CNN  | params: {sum(p.numel() for p in model_cnn.parameters()):,}')

## 5. Physiological Constraints

The model encodes cardiac physics in two complementary ways:
structural features and a single classification loss.

$$\mathcal{L} = \mathcal{L}_{\mathrm{CE}}(\hat{y}, y)$$

There are no auxiliary loss penalties. All physiological constraints
are enforced structurally through the architecture.

### 5.1 PhysioFrequencyMask — learned spectral gating

The mask filters the input signal before the FNO backbone.
Its learnable parameters are trained end-to-end via $\mathcal{L}_{\mathrm{CE}}$:

- **Per-mode logits** `logit[k]`: free gain per Fourier mode,
  initialised high inside S1/S2 bands
- **S1 band boundaries** `s1_lo`, `s1_hi`: data-driven adaptation
  of the mitral/tricuspid valve closure band
- **S2 band boundaries** `s2_lo`, `s2_hi`: data-driven adaptation
  of the aortic/pulmonary valve closure band

### 5.2 PhysioFeatureExtractor — explicit physical quantities

18 physical features are computed from the raw signal and fused
with the FNO embedding before classification.
Cross-entropy directly supervises the network to use these features:

| # | Feature | Learnable component |
|---|---------|---------------------|
| 1–2 | S1/S2 spectral energy | S1/S2 band boundaries |
| 3–4 | S1/S2 spectral centroid | S1/S2 band boundaries |
| 5 | Murmur bandwidth | Murmur band boundaries |
| 6–9 | Harmonic energies $f_0$–$4f_0$ | Harmonic weights + HR band |
| 10 | Dominant frequency (heart rate) | HR search band |
| 11 | Periodicity ratio | Fixed |
| 12 | RMS amplitude | Fixed |
| 13 | S1/S2 energy ratio | S1/S2 band boundaries |
| 14–15 | Envelope mean + std | Fixed (Hilbert transform) |
| 16 | Instantaneous frequency mean | Fixed |
| 17 | S1-to-S2 interval | HR band boundaries |
| 18 | Cross-beat consistency | HR band boundaries |

### 5.3 Why this is stronger than loss penalties

The previous design used explicit penalty terms:

$$\mathcal{L} = \mathcal{L}_{\mathrm{CE}}
+ \lambda_1 \mathcal{L}_{\mathrm{periodicity}}
+ \lambda_2 \mathcal{L}_{\mathrm{freq\ hierarchy}}$$

These penalties operate on the raw input signal and contribute
**zero gradient** to the model parameters — the signal is fixed data,
not a network output. The structural approach replaces them with
features that are direct inputs to the classifier, giving the network
an explicit, differentiable pathway from physical quantities to
classification decisions.

In [ ]:
# class PhysioFeatureExtractor(nn.Module):
#     """
#     Extracts explicit physical quantities from a raw PCG signal.

#     All quantities are differentiable w.r.t. the input signal.
#     Some have learnable parameters (band boundaries, weights).
#     Others are fixed physical computations.

#     Quantities computed:
#       1.  S1/S2 spectral energy          (2 values)
#       2.  Spectral centroid S1, S2 bands (2 values)
#       3.  Murmur bandwidth               (1 value)
#       4.  Harmonic energies f0,2f0,3f0,4f0 (4 values)
#       5.  Dominant frequency (heart rate)(1 value)
#       6.  Periodicity ratio              (1 value)
#       7.  RMS amplitude                  (1 value)
#       8.  S1/S2 energy ratio             (1 value)
#       9.  Envelope mean and std          (2 values)
#       10. Instantaneous frequency mean   (1 value)
#       11. S1-to-S2 interval estimate     (1 value)
#       12. Cross-beat consistency         (1 value)
#     ─────────────────────────────────────────────
#     Total:                               18 values
#     """
#     def __init__(self, signal_length=CONFIG['signal_length'],
#                  sr=CONFIG['sample_rate'],
#                  n_modes=CONFIG['fno_modes']):
#         super().__init__()
#         self.sr  = sr
#         self.L   = signal_length
#         nyquist  = sr / 2.0

#         # Learnable band boundaries — stored as logits, mapped to Hz via sigmoid * nyquist
#         self.s1_lo      = nn.Parameter(torch.tensor(_hz_to_logit(25,  nyquist)))
#         self.s1_hi      = nn.Parameter(torch.tensor(_hz_to_logit(45,  nyquist)))
#         self.s2_lo      = nn.Parameter(torch.tensor(_hz_to_logit(50,  nyquist)))
#         self.s2_hi      = nn.Parameter(torch.tensor(_hz_to_logit(70,  nyquist)))
#         self.mu_lo      = nn.Parameter(torch.tensor(_hz_to_logit(100, nyquist)))
#         self.mu_hi      = nn.Parameter(torch.tensor(_hz_to_logit(500, nyquist)))
#         self.hr_lo      = nn.Parameter(torch.tensor(_hz_to_logit(0.8, nyquist)))
#         self.hr_hi      = nn.Parameter(torch.tensor(_hz_to_logit(3.0, nyquist)))
#         # Learnable harmonic weights
#         self.harmonic_w = nn.Parameter(torch.ones(4))

#     # ── Helper methods ────────────────────────────────────────────────────

#     def _psd_and_freqs(self, s):
#         """PSD and frequency axis for s : (B, L)."""
#         psd = torch.fft.rfft(s, norm='ortho').abs() ** 2
#         f   = torch.fft.rfftfreq(self.L, 1.0 / self.sr).to(s.device)
#         return psd, f

#     def _band_energy(self, psd, f, lo, hi):
#         """Soft band energy. lo, hi are scalar tensors in Hz."""
#         k    = 50.0
#         mask = (torch.sigmoid(k * (f - lo)) *
#                 torch.sigmoid(k * (hi - f)))        # (F,)
#         return (psd * mask.unsqueeze(0)).sum(-1)     # (B,)

#     def _spectral_centroid(self, psd, f, lo, hi):
#         """Frequency center of mass within a soft band."""
#         k    = 50.0
#         mask = (torch.sigmoid(k * (f - lo)) *
#                 torch.sigmoid(k * (hi - f)))        # (F,)
#         num  = (psd * mask.unsqueeze(0) * f.unsqueeze(0)).sum(-1)
#         den  = (psd * mask.unsqueeze(0)).sum(-1) + 1e-8
#         return num / den                             # (B,)

#     def _hilbert_envelope(self, s):
#         """
#         Envelope via FFT-based Hilbert transform.
#         envelope = |s + i*H[s]|   fully differentiable.
#         """
#         S    = torch.fft.rfft(s, norm='ortho')       # (B, F)
#         h    = torch.zeros(S.shape[-1], device=s.device)
#         h[0] = 1.0
#         if self.L % 2 == 0:
#             h[1:self.L//2] = 2.0
#             h[self.L//2]   = 1.0
#         else:
#             h[1:(self.L+1)//2] = 2.0
#         analytic = torch.fft.irfft(S * h.unsqueeze(0), n=self.L, norm='ortho')
#         return (s ** 2 + analytic ** 2).sqrt()       # (B, L)

#     def _instantaneous_frequency(self, s):
#         """
#         Instantaneous frequency via phase derivative of analytic signal.
#         IF(t) = (1/2π) * d/dt [arg(analytic(t))]
#         """
#         S    = torch.fft.rfft(s, norm='ortho')
#         h    = torch.zeros(S.shape[-1], device=s.device)
#         h[0] = 1.0
#         if self.L % 2 == 0:
#             h[1:self.L//2] = 2.0
#             h[self.L//2]   = 1.0
#         else:
#             h[1:(self.L+1)//2] = 2.0
#         analytic  = torch.fft.irfft(S * h.unsqueeze(0), n=self.L, norm='ortho')
#         phase     = torch.atan2(analytic, s)                    # (B, L)
#         inst_freq = torch.diff(phase, dim=-1) / (2 * math.pi) * self.sr
#         return inst_freq                                         # (B, L-1)

#     def _cross_beat_consistency(self, s, psd, f):
#         """
#         Ratio of energy at f0 and first 3 harmonics to total energy.
#         High value = consistent repeating cardiac pattern.
#         Fully vectorised over batch dimension.
#         """
#         k     = 50.0
#         hr_lo = torch.sigmoid(self.hr_lo) * (self.sr / 2.0)
#         hr_hi = torch.sigmoid(self.hr_hi) * (self.sr / 2.0)

#         hr_mask = (torch.sigmoid(k * (f - hr_lo)) *
#                    torch.sigmoid(k * (hr_hi - f)))              # (F,)
#         f0_idx  = (psd * hr_mask.unsqueeze(0)).argmax(-1)       # (B,)
#         f0      = f[f0_idx]                                     # (B,)

#         width           = 2.0
#         harmonic_energy = torch.zeros(s.shape[0], device=s.device)
#         for n in range(1, 5):
#             nf0       = (n * f0).unsqueeze(1)                   # (B, 1)
#             lo_n      = nf0 - width                             # (B, 1)
#             hi_n      = nf0 + width                             # (B, 1)
#             band_mask = (torch.sigmoid(k * (f.unsqueeze(0) - lo_n)) *
#                          torch.sigmoid(k * (hi_n - f.unsqueeze(0))))  # (B, F)
#             harmonic_energy += (psd * band_mask).sum(-1)        # (B,)

#         return harmonic_energy / (psd.sum(-1) + 1e-8)           # (B,)

#     def get_bands(self):
#     """Learned band boundaries in Hz after training."""
#     nyquist = self.sr / 2.0
#     return {
#         'S1'    : [torch.sigmoid(self.s1_lo).item() * nyquist,
#                    torch.sigmoid(self.s1_hi).item() * nyquist],
#         'S2'    : [torch.sigmoid(self.s2_lo).item() * nyquist,
#                    torch.sigmoid(self.s2_hi).item() * nyquist],
#         'murmur': [torch.sigmoid(self.mu_lo).item() * nyquist,
#                    torch.sigmoid(self.mu_hi).item() * nyquist],
#         'HR'    : [torch.sigmoid(self.hr_lo).item() * nyquist,
#                    torch.sigmoid(self.hr_hi).item() * nyquist],
#     }

#     # ── Forward ───────────────────────────────────────────────────────────

#     def forward(self, x):
#         """
#         x : (B, 1, L) raw PCG signal
#         Returns: (B, 18) physical feature vector
#         """
#         s       = x.squeeze(1)                                  # (B, L)
#         psd, f  = self._psd_and_freqs(s)
#         nyquist = self.sr / 2.0
#         k       = 50.0

#         # Band boundaries in Hz
#         s1_lo = torch.sigmoid(self.s1_lo) * nyquist
#         s1_hi = torch.sigmoid(self.s1_hi) * nyquist
#         s2_lo = torch.sigmoid(self.s2_lo) * nyquist
#         s2_hi = torch.sigmoid(self.s2_hi) * nyquist
#         mu_lo = torch.sigmoid(self.mu_lo) * nyquist
#         mu_hi = torch.sigmoid(self.mu_hi) * nyquist
#         hr_lo = torch.sigmoid(self.hr_lo) * nyquist
#         hr_hi = torch.sigmoid(self.hr_hi) * nyquist

#         # 1. S1/S2 spectral energy
#         e_s1 = self._band_energy(psd, f, s1_lo, s1_hi)         # (B,)
#         e_s2 = self._band_energy(psd, f, s2_lo, s2_hi)         # (B,)

#         # 2. Spectral centroid S1, S2
#         c_s1 = self._spectral_centroid(psd, f, s1_lo, s1_hi)   # (B,)
#         c_s2 = self._spectral_centroid(psd, f, s2_lo, s2_hi)   # (B,)

#         # 3. Murmur bandwidth (spectral spread in murmur band)
#         mu_e    = self._band_energy(psd, f, mu_lo, mu_hi)
#         mu_c    = self._spectral_centroid(psd, f, mu_lo, mu_hi)
#         mu_mask = (torch.sigmoid(k * (f - mu_lo)) *
#                    torch.sigmoid(k * (mu_hi - f)))
#         mu_var  = ((psd * mu_mask.unsqueeze(0) *
#                     (f.unsqueeze(0) - mu_c.unsqueeze(1)) ** 2).sum(-1) /
#                    (mu_e + 1e-8))
#         mu_bw   = mu_var.sqrt()                                  # (B,)

#         # 4. Harmonic energies at f0, 2f0, 3f0, 4f0 — fully vectorised
#         hr_mask = (torch.sigmoid(k * (f - hr_lo)) *
#                    torch.sigmoid(k * (hr_hi - f)))
#         f0_idx  = (psd * hr_mask.unsqueeze(0)).argmax(-1)       # (B,)
#         f0_hz   = f[f0_idx]                                     # (B,)
#         width   = 2.0
#         harm_energies = []
#         for n in range(1, 5):
#             nf0       = (n * f0_hz).unsqueeze(1)                # (B, 1)
#             lo_n      = nf0 - width                             # (B, 1)
#             hi_n      = nf0 + width                             # (B, 1)
#             band_mask = (torch.sigmoid(k * (f.unsqueeze(0) - lo_n)) *
#                          torch.sigmoid(k * (hi_n - f.unsqueeze(0))))  # (B, F)
#             harm_energies.append((psd * band_mask).sum(-1))     # (B,)
#         harm_energies = torch.stack(harm_energies, dim=1)       # (B, 4)

#         # 5. Dominant frequency (heart rate estimate)
#         f0_feat = f0_hz / nyquist                               # (B,) normalised

#         # 6. Periodicity ratio
#         band   = (f >= 1.0) & (f <= 80.0 / 60 * 4)
#         period = psd[:, band].sum(-1) / (psd.sum(-1) + 1e-8)   # (B,)

#         # 7. RMS amplitude
#         rms = (s ** 2).mean(-1).sqrt()                          # (B,)

#         # 8. S1/S2 energy ratio (log scale)
#         ratio = e_s1 / (e_s2 + 1e-8)                           # (B,)

#         # 9. Envelope mean and std
#         env      = self._hilbert_envelope(s)                    # (B, L)
#         env_mean = env.mean(-1)                                  # (B,)
#         env_std  = env.std(-1)                                   # (B,)

#         # 10. Instantaneous frequency mean (normalised)
#         if_sig  = self._instantaneous_frequency(s)              # (B, L-1)
#         if_mean = if_sig.mean(-1) / nyquist                     # (B,)

#         # 11. S1-to-S2 interval estimate: 1 / (2 * f0)
#         s1s2_interval = 1.0 / (2.0 * f0_hz.clamp(min=0.5))     # (B,) seconds

#         # 12. Cross-beat consistency
#         consistency = self._cross_beat_consistency(s, psd, f)   # (B,)

#         # Stack → (B, 18)
#         features = torch.stack([
#             e_s1, e_s2,                                # 1. spectral energy
#             c_s1 / nyquist, c_s2 / nyquist,            # 2. centroid
#             mu_bw / nyquist,                            # 3. murmur bandwidth
#             harm_energies[:, 0], harm_energies[:, 1],
#             harm_energies[:, 2], harm_energies[:, 3],  # 4. harmonics
#             f0_feat,                                   # 5. dominant frequency
#             period,                                    # 6. periodicity
#             rms,                                       # 7. RMS
#             ratio.log1p(),                             # 8. S1/S2 ratio
#             env_mean, env_std,                         # 9. envelope
#             if_mean,                                   # 10. inst. frequency
#             s1s2_interval,                             # 11. S1-S2 interval
#             consistency,                               # 12. cross-beat
#         ], dim=1)                                      # (B, 18)

#         return features


# def total_loss(logits, targets, signal, model):
#     lce = F.cross_entropy(logits, targets)
#     return {
#         'total'        : lce,
#         'cross_entropy': lce.item(),
#         'periodicity'  : 0.0,
#         'frequency'    : 0.0,
#     }

## 6. Training

In [ ]:
def train_model(model, train_loader, val_loader,
                model_name='model', use_physio=True,
                epochs=CONFIG['epochs']):

    opt  = torch.optim.AdamW(model.parameters(),
                              lr=CONFIG['lr'],
                              weight_decay=CONFIG['weight_decay'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    hist = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc',
                             'loss_ce','loss_period','loss_freq']}
    best_val = 0.0

    for ep in range(1, epochs+1):
        model.train()
        run = dict(total=0., ce=0., period=0., freq=0.)
        correct = total = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            if use_physio:
                logits, _ = model(xb)
                losses    = total_loss(logits, yb, xb, model)
                loss      = losses['total']
                run['ce']     += losses['cross_entropy']
                run['period'] += losses['periodicity'] if isinstance(losses['periodicity'], float) else losses['periodicity'].item()
                run['freq']   += losses['frequency']   if isinstance(losses['frequency'],   float) else losses['frequency'].item()
            else:
                out    = model(xb)
                logits = out[0] if isinstance(out, tuple) else out
                loss   = F.cross_entropy(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            run['total'] += loss.item()
            correct += (logits.argmax(1) == yb).sum().item()
            total   += yb.size(0)

        sched.step()
        nb = len(train_loader)
        train_acc, train_loss = correct/total, run['total']/nb

        model.eval()
        vc = vt = vl = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out    = model(xb)
                logits = out[0] if isinstance(out, tuple) else out
                vl += F.cross_entropy(logits, yb).item()
                vc += (logits.argmax(1) == yb).sum().item()
                vt += yb.size(0)
        val_acc, val_loss = vc/vt, vl/len(val_loader)

        for k, v in [('train_loss',train_loss),('val_loss',val_loss),
                     ('train_acc',train_acc),  ('val_acc', val_acc),
                     ('loss_ce',   run['ce']/nb),
                     ('loss_period',run['period']/nb),
                     ('loss_freq',  run['freq']/nb)]:
            hist[k].append(v)

        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(),
                       f'checkpoints/{model_name}_best.pt')

        if ep % 10 == 0 or ep == 1:
            print(f'[{model_name}] Ep {ep:3d}/{epochs}  '
                  f'train acc {train_acc:.3f}  val acc {val_acc:.3f}  '
                  f'loss {train_loss:.4f}')

    model.load_state_dict(
        torch.load(f'checkpoints/{model_name}_best.pt', map_location=DEVICE))
    print(f'[{model_name}] Best val acc: {best_val:.4f}')
    return hist


print('Training FNO (physiologically constrained)...')
history_fno = train_model(model_fno, train_loader, val_loader,
                           model_name='fno', use_physio=True)

print('\nTraining CNN baseline...')
history_cnn = train_model(model_cnn, train_loader, val_loader,
                           model_name='cnn', use_physio=False)

## 7. Training Curves

In [ ]:
ep  = range(1, CONFIG['epochs']+1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(ep, history_fno['train_acc'], 'b-',  label='FNO train')
axes[0].plot(ep, history_fno['val_acc'],   'b--', label='FNO val')
axes[0].plot(ep, history_cnn['train_acc'], 'r-',  label='CNN train')
axes[0].plot(ep, history_cnn['val_acc'],   'r--', label='CNN val')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history_fno['train_loss'], 'b-',  label='FNO train')
axes[1].plot(ep, history_fno['val_loss'],   'b--', label='FNO val')
axes[1].plot(ep, history_cnn['train_loss'], 'r-',  label='CNN train')
axes[1].plot(ep, history_cnn['val_loss'],   'r--', label='CNN val')
axes[1].set_title('Total Loss'); axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(ep, history_fno['loss_ce'],     label='Cross-entropy', color='navy')
axes[2].plot(ep, history_fno['loss_period'], label='Periodicity',   color='darkorange')
axes[2].plot(ep, history_fno['loss_freq'],   label='Freq hierarchy',color='darkgreen')
axes[2].set_title('FNO Loss Components\n(period & freq = 0: now structural constraints)')
axes[2].grid(alpha=0.3)

for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Test Set Evaluation

In [ ]:
def evaluate(model, loader, model_name):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            out    = model(xb.to(DEVICE))
            logits = out[0] if isinstance(out, tuple) else out
            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(yb.numpy())
    acc = np.mean(np.array(preds) == np.array(labels))
    print(f'\n{"="*52}')
    print(f'{model_name}  |  Test Accuracy: {acc:.4f}')
    print('='*52)
    print(classification_report(labels, preds,
                                 target_names=CONFIG['class_names']))
    return preds, labels, acc


preds_fno, labels_fno, acc_fno = evaluate(
    model_fno, test_loader, 'FNO (physio-constrained)')
preds_cnn, labels_cnn, acc_cnn = evaluate(
    model_cnn, test_loader, 'CNN Baseline')

print(f'\nAccuracy delta: {(acc_fno - acc_cnn)*100:+.1f} percentage points')

# ── Learned band boundaries ───────────────────────────────────────────────
print('\nLearned band boundaries after training:')

mask_bands = model_fno.freq_mask.get_bands()
print(f'  PhysioFrequencyMask:')
print(f'    S1 : {mask_bands["S1"][0]:.1f} – {mask_bands["S1"][1]:.1f} Hz'
      f'  (init: 25.0 – 45.0 Hz)')
print(f'    S2 : {mask_bands["S2"][0]:.1f} – {mask_bands["S2"][1]:.1f} Hz'
      f'  (init: 50.0 – 70.0 Hz)')

phys_bands = model_fno.phys.get_bands()
print(f'  PhysioFeatureExtractor:')
print(f'    S1     : {phys_bands["S1"][0]:.1f} – {phys_bands["S1"][1]:.1f} Hz'
      f'  (init: 25.0 – 45.0 Hz)')
print(f'    S2     : {phys_bands["S2"][0]:.1f} – {phys_bands["S2"][1]:.1f} Hz'
      f'  (init: 50.0 – 70.0 Hz)')
print(f'    Murmur : {phys_bands["murmur"][0]:.1f} – {phys_bands["murmur"][1]:.1f} Hz'
      f'  (init: 100.0 – 500.0 Hz)')
print(f'    HR     : {phys_bands["HR"][0]:.2f} – {phys_bands["HR"][1]:.2f} Hz'
      f'  (init: 0.80 – 3.00 Hz)')

## 9. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, p, l, title in [
    (axes[0], preds_fno, labels_fno,
     f'FNO (physio-constrained)\nAcc = {acc_fno:.3f}'),
    (axes[1], preds_cnn, labels_cnn,
     f'CNN Baseline\nAcc = {acc_cnn:.3f}'),
]:
    cm = confusion_matrix(l, p, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', ax=ax,
                xticklabels=CONFIG['class_names'],
                yticklabels=CONFIG['class_names'],
                cmap='Blues', vmin=0, vmax=1)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Learned physical quantities

In [ ]:
fig = plt.figure(figsize=(18, 20))
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('Learned Physical Quantities After Training',
             fontsize=14, fontweight='bold', y=0.98)

nyq        = CONFIG['sample_rate'] / 2.0
mask_bands = model_fno.freq_mask.get_bands()
phys_bands = model_fno.phys.get_bands()

# ── 1. Learned frequency mask (PhysioFrequencyMask) ──────────────────────
ax1 = fig.add_subplot(gs[0, :2])
mask  = model_fno.freq_mask.get_mask()
freqs = np.fft.rfftfreq(CONFIG['signal_length'],
                         1.0/CONFIG['sample_rate'])[:CONFIG['fno_modes']]
dfreq = freqs[1] - freqs[0]
ax1.bar(freqs, mask, width=dfreq, color='steelblue', alpha=0.8)
ax1.axvspan(*mask_bands['S1'], alpha=0.2, color='royalblue',
            label=f'S1 learned ({mask_bands["S1"][0]:.1f}–{mask_bands["S1"][1]:.1f} Hz)')
ax1.axvspan(*mask_bands['S2'], alpha=0.2, color='tomato',
            label=f'S2 learned ({mask_bands["S2"][0]:.1f}–{mask_bands["S2"][1]:.1f} Hz)')
ax1.set_title('PhysioFrequencyMask — learned per-mode gains')
ax1.set_xlabel('Frequency (Hz)'); ax1.set_ylabel('Mask weight')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# ── 2. Band boundary drift: init vs learned ───────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
bands_init   = {'S1': [25, 45],   'S2': [50, 70]}
band_labels  = ['S1 lo', 'S1 hi', 'S2 lo', 'S2 hi']
mask_vals    = [mask_bands['S1'][0], mask_bands['S1'][1],
                mask_bands['S2'][0], mask_bands['S2'][1]]
phys_vals    = [phys_bands['S1'][0], phys_bands['S1'][1],
                phys_bands['S2'][0], phys_bands['S2'][1]]
init_vals    = [25, 45, 50, 70]
x_pos        = np.arange(4)
ax2.bar(x_pos - 0.25, init_vals,  0.25, label='Init',    color='gray',      alpha=0.7)
ax2.bar(x_pos,        mask_vals,  0.25, label='Mask',    color='steelblue', alpha=0.8)
ax2.bar(x_pos + 0.25, phys_vals,  0.25, label='Extractor', color='tomato',  alpha=0.8)
ax2.set_xticks(x_pos); ax2.set_xticklabels(band_labels, fontsize=8)
ax2.set_ylabel('Hz'); ax2.set_title('S1/S2 boundary drift\n(init vs learned)')
ax2.legend(fontsize=7); ax2.grid(alpha=0.3, axis='y')

# ── 3. PhysioFeatureExtractor — S1/S2 bands ──────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
all_freqs = np.fft.rfftfreq(CONFIG['signal_length'], 1.0/CONFIG['sample_rate'])
k = 50.0
s1_bump = (1 / (1 + np.exp(-k*(all_freqs - phys_bands['S1'][0]))) *
           1 / (1 + np.exp(-k*(phys_bands['S1'][1] - all_freqs))))
s2_bump = (1 / (1 + np.exp(-k*(all_freqs - phys_bands['S2'][0]))) *
           1 / (1 + np.exp(-k*(phys_bands['S2'][1] - all_freqs))))
ax3.fill_between(all_freqs, s1_bump, alpha=0.6, color='royalblue', label='S1 bump')
ax3.fill_between(all_freqs, s2_bump, alpha=0.6, color='tomato',    label='S2 bump')
ax3.set_xlim(0, 150); ax3.set_xlabel('Frequency (Hz)')
ax3.set_ylabel('Soft indicator'); ax3.grid(alpha=0.3)
ax3.set_title('Extractor S1/S2 soft bands'); ax3.legend(fontsize=8)

# ── 4. Murmur band ────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
mu_bump = (1 / (1 + np.exp(-k*(all_freqs - phys_bands['murmur'][0]))) *
           1 / (1 + np.exp(-k*(phys_bands['murmur'][1] - all_freqs))))
ax4.fill_between(all_freqs, mu_bump, alpha=0.6, color='darkorange')
ax4.axvline(phys_bands['murmur'][0], color='darkorange', ls='--', lw=1,
            label=f'lo={phys_bands["murmur"][0]:.0f} Hz')
ax4.axvline(phys_bands['murmur'][1], color='darkorange', ls='-',  lw=1,
            label=f'hi={phys_bands["murmur"][1]:.0f} Hz')
ax4.set_xlabel('Frequency (Hz)'); ax4.set_ylabel('Soft indicator')
ax4.set_title('Extractor murmur band\n(init: 100–500 Hz)')
ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

# ── 5. Heart rate search band ─────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
hr_bump = (1 / (1 + np.exp(-k*(all_freqs - phys_bands['HR'][0]))) *
           1 / (1 + np.exp(-k*(phys_bands['HR'][1] - all_freqs))))
ax5.fill_between(all_freqs, hr_bump, alpha=0.6, color='seagreen')
ax5.axvline(phys_bands['HR'][0], color='seagreen', ls='--', lw=1,
            label=f'lo={phys_bands["HR"][0]:.2f} Hz')
ax5.axvline(phys_bands['HR'][1], color='seagreen', ls='-',  lw=1,
            label=f'hi={phys_bands["HR"][1]:.2f} Hz')
ax5.set_xlim(0, 10); ax5.set_xlabel('Frequency (Hz)')
ax5.set_ylabel('Soft indicator')
ax5.set_title('Extractor HR search band\n(init: 0.8–3.0 Hz)')
ax5.legend(fontsize=8); ax5.grid(alpha=0.3)

# ── 6. Harmonic weights ───────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 0])
hw  = torch.softmax(model_fno.phys.harmonic_w, dim=0).detach().cpu().numpy()
ax6.bar(['f₀', '2f₀', '3f₀', '4f₀'], hw,
        color=['steelblue','royalblue','cornflowerblue','lightsteelblue'],
        alpha=0.85, edgecolor='navy')
ax6.axhline(0.25, color='gray', ls='--', lw=1, label='Uniform (0.25)')
ax6.set_ylabel('Softmax weight'); ax6.set_ylim(0, 1)
ax6.set_title('Learned harmonic weights\n(which harmonics matter most)')
ax6.legend(fontsize=8); ax6.grid(alpha=0.3, axis='y')
for i, v in enumerate(hw):
    ax6.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

# ── 7. SpectralConv1d weight magnitudes (first FNO block) ─────────────────
ax7 = fig.add_subplot(gs[2, 1])
W_complex = torch.complex(model_fno.fno_blocks[0].spec.wr,
                           model_fno.fno_blocks[0].spec.wi)  # (in,out,modes)
W_mag = W_complex.abs().mean(dim=(0,1)).detach().cpu().numpy()  # (modes,)
mode_idx = np.arange(len(W_mag))
ax7.bar(mode_idx, W_mag, color='purple', alpha=0.7)
ax7.set_xlabel('Fourier mode index'); ax7.set_ylabel('Mean |W|')
ax7.set_title('SpectralConv1d weight magnitudes\n(FNO block 1, averaged over channels)')
ax7.grid(alpha=0.3)

# ── 8. SpectralConv1d weight magnitudes across all FNO blocks ─────────────
ax8 = fig.add_subplot(gs[2, 2])
colors_blk = plt.cm.viridis(np.linspace(0, 1, CONFIG['fno_depth']))
for i, blk in enumerate(model_fno.fno_blocks):
    W  = torch.complex(blk.spec.wr, blk.spec.wi)
    Wm = W.abs().mean(dim=(0,1)).detach().cpu().numpy()
    ax8.plot(mode_idx, Wm, color=colors_blk[i], lw=1.5,
             label=f'Block {i+1}')
ax8.set_xlabel('Fourier mode index'); ax8.set_ylabel('Mean |W|')
ax8.set_title('SpectralConv1d magnitudes\nacross all FNO blocks')
ax8.legend(fontsize=8); ax8.grid(alpha=0.3)

# ── 9. Per-mode logit values (raw, before sigmoid) ────────────────────────
ax9 = fig.add_subplot(gs[3, 0])
logits_raw = model_fno.freq_mask.logit.detach().cpu().numpy()
ax9.bar(freqs, logits_raw, width=dfreq, color='darkcyan', alpha=0.8)
ax9.axhline(0, color='black', lw=0.8, ls='--')
ax9.set_xlabel('Frequency (Hz)'); ax9.set_ylabel('Logit value')
ax9.set_title('Per-mode logits (raw)\n>0 → gain>0.5, <0 → gain<0.5')
ax9.grid(alpha=0.3)

# ── 10. Mask comparison: init vs learned ──────────────────────────────────
ax10 = fig.add_subplot(gs[3, 1])
mask_init_vals = np.full(len(freqs), 0.1)
s1_init = (freqs >= 25) & (freqs <= 45)
s2_init = (freqs >= 50) & (freqs <= 70)
mask_init_vals[s1_init | s2_init] = 1.0
ax10.plot(freqs, mask_init_vals, 'k--', lw=1.5, label='Init', alpha=0.6)
ax10.plot(freqs, mask,           'b-',  lw=1.5, label='Learned')
ax10.fill_between(freqs, mask_init_vals, mask, alpha=0.15, color='blue',
                  label='Δ mask')
ax10.set_xlabel('Frequency (Hz)'); ax10.set_ylabel('Mask weight')
ax10.set_title('Mask: init vs learned\n(blue fill = change)')
ax10.legend(fontsize=8); ax10.grid(alpha=0.3)

# ── 11. Summary table ─────────────────────────────────────────────────────
ax11 = fig.add_subplot(gs[3, 2])
ax11.axis('off')
rows = [
    ['Parameter', 'Init', 'Learned'],
    ['Mask S1 lo', '25.0 Hz', f'{mask_bands["S1"][0]:.1f} Hz'],
    ['Mask S1 hi', '45.0 Hz', f'{mask_bands["S1"][1]:.1f} Hz'],
    ['Mask S2 lo', '50.0 Hz', f'{mask_bands["S2"][0]:.1f} Hz'],
    ['Mask S2 hi', '70.0 Hz', f'{mask_bands["S2"][1]:.1f} Hz'],
    ['Phys S1 lo', '25.0 Hz', f'{phys_bands["S1"][0]:.1f} Hz'],
    ['Phys S1 hi', '45.0 Hz', f'{phys_bands["S1"][1]:.1f} Hz'],
    ['Phys S2 lo', '50.0 Hz', f'{phys_bands["S2"][0]:.1f} Hz'],
    ['Phys S2 hi', '70.0 Hz', f'{phys_bands["S2"][1]:.1f} Hz'],
    ['Murmur lo',  '100 Hz',  f'{phys_bands["murmur"][0]:.0f} Hz'],
    ['Murmur hi',  '500 Hz',  f'{phys_bands["murmur"][1]:.0f} Hz'],
    ['HR lo',      '0.80 Hz', f'{phys_bands["HR"][0]:.2f} Hz'],
    ['HR hi',      '3.00 Hz', f'{phys_bands["HR"][1]:.2f} Hz'],
    ['harm w f₀',  '0.250',   f'{hw[0]:.3f}'],
    ['harm w 2f₀', '0.250',   f'{hw[1]:.3f}'],
    ['harm w 3f₀', '0.250',   f'{hw[2]:.3f}'],
    ['harm w 4f₀', '0.250',   f'{hw[3]:.3f}'],
]
tbl = ax11.table(cellText=rows[1:], colLabels=rows[0],
                  loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(7)
tbl.scale(1.1, 1.25)
ax11.set_title('All learned quantities\nsummary', fontweight='bold', fontsize=9)

plt.savefig('figures/learned_physical_quantities.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 11. Latent Space (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

model_fno.eval()
embs, tlabels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        _, emb = model_fno(xb.to(DEVICE))
        embs.append(emb.cpu().numpy())
        tlabels.extend(yb.numpy())

embs    = np.concatenate(embs)
tlabels = np.array(tlabels)
emb_2d  = TSNE(n_components=2, perplexity=15,
                random_state=42).fit_transform(embs)

fig, ax = plt.subplots(figsize=(8, 6))
for cls in range(CONFIG['n_classes']):
    idx = tlabels == cls
    ax.scatter(emb_2d[idx, 0], emb_2d[idx, 1],
               label=CONFIG['class_names'][cls],
               alpha=0.75, s=45, edgecolors='none')
ax.set_title('t-SNE  —  FNO Latent Embeddings (test set)')
ax.legend(fontsize=9); ax.axis('off')
plt.tight_layout()
plt.savefig('figures/tsne_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Summary

In [ ]:
print('\n' + '='*60)
print(' RESULTS SUMMARY')
print('='*60)
print(f' {"Model":<38} {"Test Acc":>8}')
print('-'*60)
print(f' {"CNN Baseline (Mel spectrogram)":<38} {acc_cnn:>8.4f}')
print(f' {"FNO (physio-constrained)":<38} {acc_fno:>8.4f}')
print('='*60)
print()
print(f' Classes  : AS / MR / MS / MVP / Normal')
print(f' N total  : {len(dataset)}')
print(f' SR       : {CONFIG["sample_rate"]} Hz')
print(f' Window   : {CONFIG["signal_length"]/CONFIG["sample_rate"]:.1f} s '
      f'({CONFIG["signal_length"]} samples)')
print()
print(' Physics in FNO (structural — not loss penalties):')
print()
print('  PhysioFrequencyMask:')
print('   [1] Per-mode learnable gain      free logit per Fourier mode')
print('   [2] Learnable S1 band            boundaries adapt during training')
print('   [3] Learnable S2 band            boundaries adapt during training')
print()
print('  PhysioFeatureExtractor (18 features fused with FNO embedding):')
print('   [4] S1/S2 spectral energy        learnable band boundaries')
print('   [5] S1/S2 spectral centroid      center of mass per band')
print('   [6] Murmur bandwidth             spectral spread, learnable band')
print('   [7] Harmonic energies f0–4f0     learnable harmonic weights')
print('   [8] Dominant frequency           heart rate estimate')
print('   [9] Periodicity ratio            energy in cardiac band')
print('  [10] RMS amplitude                signal energy')
print('  [11] S1/S2 energy ratio           hierarchy (structural)')
print('  [12] Envelope mean + std          Hilbert transform')
print('  [13] Instantaneous frequency      phase derivative')
print('  [14] S1-to-S2 interval            systolic duration proxy')
print('  [15] Cross-beat consistency       harmonic energy ratio')
print()
print(' Learned band boundaries after training:')
mask_bands = model_fno.freq_mask.get_bands()
phys_bands = model_fno.phys.get_bands()
print(f'  Mask  S1: {mask_bands["S1"][0]:.1f}–{mask_bands["S1"][1]:.1f} Hz  '
      f'S2: {mask_bands["S2"][0]:.1f}–{mask_bands["S2"][1]:.1f} Hz  '
      f'(init: 25–45 / 50–70 Hz)')
print(f'  Phys  S1: {phys_bands["S1"][0]:.1f}–{phys_bands["S1"][1]:.1f} Hz  '
      f'S2: {phys_bands["S2"][0]:.1f}–{phys_bands["S2"][1]:.1f} Hz  '
      f'(init: 25–45 / 50–70 Hz)')
print(f'  Murmur  : {phys_bands["murmur"][0]:.0f}–{phys_bands["murmur"][1]:.0f} Hz  '
      f'(init: 100–500 Hz)')
print(f'  HR band : {phys_bands["HR"][0]:.2f}–{phys_bands["HR"][1]:.2f} Hz  '
      f'(init: 0.80–3.00 Hz)')
hw = torch.softmax(model_fno.phys.harmonic_w, dim=0).detach().cpu().numpy()
print(f'  Harmonic weights: f0={hw[0]:.3f}  2f0={hw[1]:.3f}  '
      f'3f0={hw[2]:.3f}  4f0={hw[3]:.3f}  (init: 0.250 each)')
print('='*60)